# Gold · Dimensión de Fecha

## Propósito

Construir la dimensión calendario de la capa Gold, que permite analizar pedidos y envíos por periodo.

## Flujo

1. Calcular el rango real de fechas a partir de los pedidos de la capa Silver (primera y última fecha).
2. Construir un registro por día dentro de ese rango, con sus atributos de calendario.
3. Escribir la tabla en Gold.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_dim_fecha`

**Llave:** `fecha_key`

## Atributos generados

* Año, mes y día
* `es_fin_de_semana`

## Relación con los hechos

La fecha de pedido y la fecha de envío de los hechos de pedidos se relacionan con esta dimensión.

## Resultado esperado

Un calendario continuo que cubre todo el rango de fechas con datos, listo para el análisis por periodo.

In [0]:
# ============================================================
# DIM_FECHA — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"
GOLD_DIM_FECHA_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_fecha"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_PEDIDOS_FQN}")
print(f"✓ Destino: {GOLD_DIM_FECHA_FQN}")

In [0]:
# ============================================================
# DIM_FECHA — Celda 2: INICIO DE LOG DE EJECUCIÓN — gld_dim_fecha
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_dim_fecha"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# DIM_FECHA — Celda 3: Rango de fechas real (desde Pedidos)
# ============================================================

rango = spark.sql(f"""
    SELECT
        MIN(LEAST(order_date, shipping_date)) AS fecha_min,
        MAX(GREATEST(order_date, shipping_date)) AS fecha_max
    FROM {SILVER_PEDIDOS_FQN}
""").collect()[0]

fecha_min = rango["fecha_min"]
fecha_max = rango["fecha_max"]

print(f"✓ Rango detectado: {fecha_min} → {fecha_max}")

In [0]:
# ============================================================
# DIM_FECHA — Celda 4: Construcción del calendario
# ============================================================

from pyspark.sql import functions as F

MESES_ES = [
    "", "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

DIAS_ES = [
    "", "Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"
]

df_calendario = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{fecha_min}'),
        to_date('{fecha_max}'),
        interval 1 day
    )) AS fecha
""")

df_dim_fecha = (
    df_calendario
    .withColumn("fecha_key", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", F.element_at(F.array(*[F.lit(m) for m in MESES_ES]), F.col("mes") + 1))
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("dia_semana_num", F.dayofweek("fecha"))
    .withColumn("nombre_dia", F.element_at(F.array(*[F.lit(d) for d in DIAS_ES]), F.col("dia_semana_num") + 1))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("semana_anio", F.weekofyear("fecha"))
    .withColumn("es_fin_de_semana", F.col("dia_semana_num").isin([1, 7]))
)

print(f"✓ Filas generadas: {df_dim_fecha.count()}")
display(df_dim_fecha.limit(5))

In [0]:
# ============================================================
# DIM_FECHA — Celda 5: Escritura en Gold
# ============================================================

(
    df_dim_fecha.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_DIM_FECHA_FQN)
)

print(f"✓ Tabla {GOLD_DIM_FECHA_FQN} escrita: {df_dim_fecha.count()} filas")

In [0]:
TABLA_DESTINO = "gld_dataco.supply_chain.desc_dim_fecha"   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")